# Local 04: Upload Fine-tuned Model to Hugging Face

로컬 환경에서 학습된 체크포인트를 Hugging Face model repo로 업로드합니다.



In [ ]:
!pip -q install -U huggingface_hub


## 1) 설정 + 함수 정의


In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, login, upload_folder

CANDIDATE_ROOTS = [
    r'C:\Users\User\Desktop\LLM_JSSP_masking',
    '/mnt/c/Users/User/Desktop/LLM_JSSP_masking',
]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if Path(p).exists()), CANDIDATE_ROOTS[-1])
PROJECT_ROOT = str(Path(PROJECT_ROOT).expanduser().resolve())
root = Path(PROJECT_ROOT)
assert root.exists(), f'프로젝트 경로 없음: {root}'

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')
MODEL_ROLE = 'policy'  # policy | reason | mixed
MODEL_REPO_MAP = {
    'policy': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'reason': 'HYUNJINI/jssp_reason_llama8b_step_r64_ep2',
    'mixed': 'HYUNJINI/jssp_mixed_llama8b_step_action_reason_r64_ep2_full_v1',
}
MODEL_PATH_MAP = {
    'policy': root / 'finetuned_models/jssp_policy_llama8b_step_r64_ep2/checkpoint-200',
    'reason': root / 'finetuned_models/jssp_reason_llama8b_step_r64_ep2/checkpoint-200',
    'mixed': root / 'finetuned_models/jssp_mixed_llama8b_step_action_reason_r64_ep2_full_v1/checkpoint-200',
}
if MODEL_ROLE not in MODEL_REPO_MAP:
    raise ValueError(f'Unsupported MODEL_ROLE={MODEL_ROLE}')
HF_MODEL_REPO = MODEL_REPO_MAP[MODEL_ROLE]
MODEL_PATH = MODEL_PATH_MAP[MODEL_ROLE]

api = HfApi(token=HF_TOKEN)


def create_hf_repo(api, repo_name, token):
    try:
        print(f"Path: 저장소 '{repo_name}' 생성/확인 중...")
        api.create_repo(repo_id=repo_name, repo_type='model', token=token, private=False, exist_ok=True)
        print(f"Success: 저장소 '{repo_name}' 준비 완료")
        return True
    except Exception as e:
        print(f"Error: 저장소 생성 중 오류: {e}")
        return False


print('PROJECT_ROOT:', root)
print('MODEL_ROLE:', MODEL_ROLE)
print('MODEL_PATH:', MODEL_PATH)
print('HF_MODEL_REPO:', HF_MODEL_REPO)


def upload_model_to_hf():
    if not create_hf_repo(api, HF_MODEL_REPO, HF_TOKEN):
        return False
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f'모델 경로 없음: {MODEL_PATH}')
    upload_folder(
        repo_id=HF_MODEL_REPO,
        repo_type='model',
        folder_path=str(MODEL_PATH),
        path_in_repo=MODEL_PATH.name,
        token=HF_TOKEN,
        commit_message=f'Upload {MODEL_ROLE} model from local folder',
    )
    return True


## 2) 업로드 실행


In [ ]:
ok = upload_model_to_hf()
if ok:
    print('Success: 업로드 완료')
else:
    print('Error: 업로드 실패')



## 3) 업로드 검증


In [ ]:
files = api.list_repo_files(repo_id=HF_MODEL_REPO, repo_type='model')
print(f'{HF_MODEL_REPO} ({len(files)} files)')
for f in files:
    print(' -', f)

